# 🧠 Module 2 – Session 2 Assignment
# The Decoding Playbook

> **Goal:** Learn how an LLM engineer chooses decoding parameters for different business tasks.

---

## Before You Start

This assignment is **not** asking you to discover the mathematically best temperature.

Instead, imagine you joined a company and your manager asks:

> **"Which decoding settings should we use for each feature, and why?"**

Your job is to justify your engineering decisions with experiments.


# 📖 Story

You work at **Lumen Desk**.

The company has three AI products.

| Product | What the model should do |
|---|---|
| Ticket Tagger | Return ONE category only |
| Reply Drafter | Write a professional reply |
| Campaign Brainstormer | Generate creative marketing ideas |

Notice that these products have **different business goals**, so they probably need **different decoding strategies**.


# 🚀 Roadmap

You will repeat the same workflow three times.

```text
Understand the task
        ↓
Define success
        ↓
Design 3 configurations
        ↓
Run experiments
        ↓
Compare results
        ↓
Choose the winner
        ↓
Write your engineering recommendation
```


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Load model and tokenizer
# Using a small pre-trained GPT-2 model for demonstration.
model_name = "gpt2"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Add a pad token if not present, needed for some models with custom generation loops
if tok.pad_token is None:
    tok.add_special_tokens({'pad_token': tok.eos_token})
    model.resize_token_embeddings(len(tok))

# Helper functions for decoding (applied to probabilities after softmax)
def apply_top_k_filtering(probs, k):
    if k == 0 or k >= probs.shape[-1]: # No filtering if k is 0 or larger than vocab size
        return probs

    top_k_values, top_k_indices = torch.topk(probs, k=k)
    filtered_probs = torch.zeros_like(probs)
    filtered_probs.scatter_(dim=-1, index=top_k_indices, src=top_k_values)
    return filtered_probs / filtered_probs.sum(dim=-1, keepdim=True) # Re-normalize

def apply_top_p_filtering(probs, p):
    if p == 0.0 or p >= 1.0: # No filtering if p is 0 or 1
        return probs

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Remove tokens with cumulative probability above p
    sorted_indices_to_remove = cumulative_probs > p
    # Keep the first token above p, even if its cumulative probability is > p
    if sorted_indices_to_remove.any():
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0

    indices_to_remove = sorted_indices[sorted_indices_to_remove]
    probs[indices_to_remove] = 0.0
    return probs / probs.sum(dim=-1, keepdim=True) # Re-normalize

# Modified generation function to include decoding parameters
def generate_with_params(prompt, n_tokens=5, temperature=1.0, top_k=0, top_p=0.0):
    ids_ = tok.encode(prompt, return_tensors="pt")
    generated_ids = ids_.clone()

    for _ in range(n_tokens):
        with torch.no_grad():
            outputs = model(generated_ids)
            logits = outputs.logits[0, -1] # Get logits for the last token

        # Apply temperature and get probabilities
        if temperature == 0.0: # Greedy decoding: directly take argmax
            next_token_id = torch.argmax(logits, dim=-1).unsqueeze(0)
        else:
            probs = torch.softmax(logits / temperature, dim=-1)

            # Apply top-k filtering
            if top_k > 0:
                probs = apply_top_k_filtering(probs, top_k)

            # Apply top-p filtering
            if top_p > 0.0:
                probs = apply_top_p_filtering(probs, top_p)

            # If all probabilities are zero after filtering (e.g., due to extreme filtering or very low values),
            # fall back to greedy to avoid errors.
            if probs.sum() == 0:
                next_token_id = torch.argmax(logits, dim=-1).unsqueeze(0)
            else:
                next_token_id = torch.multinomial(probs, 1)

        generated_ids = torch.cat([generated_ids, next_token_id.view(1, 1)], dim=1)

    return tok.decode(generated_ids[0])


# Ticket Tagger

## Step 1 — Understand the Business Problem

### Success Criteria

**Exactly one category, deterministic, strict format.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
2. Is creativity helpful or harmful?
3. Should the model take risks?

Write your answers below.
1- yes it should be always identical because the goal and sucess criteria is one category and deterministic

2-creativity is harmful here becaues as i saud the goal should be in strict format, creativity would introduce unwanted variations

3-no because the model should be deterministic, risks could lead to ambiguity or non-compliance wiith the required format

---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---|---|---|
|Primary|0.0|1|0.0|0.0|Greedy|Output must be deterministic and in a strict format, with exactly one category. Greedy decoding with zero temperature, top-k, and top-p ensures the model picks the most probable token every time, leading to consistent, non-creative, and predictable output.|
|Challenger A|0.1|5|0.1|0.0|Sampling|Slightly more flexible than Primary but still highly constrained. A small temperature and limited top-k/top-p allow for minor variations, which might still adhere to the strict format but could introduce some unexpected classifications if the probabilities are very close.|
|Challenger B|0.5|40|0.9|0.0|Sampling|Introduces more randomness and creativity to demonstrate the harm for a classification task. A higher temperature and broader top-k/top-p mean the model explores more options, increasing the risk of non-deterministic, inconsistent, or even multi-category outputs, which violates the success criteria.|


---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
Classify the following customer support ticket into one of these categories: 'Billing', 'Technical Support', 'Account Management', 'General Inquiry'.

Ticket: My internet isn't working after the update. I can't connect to any websites.
Category:```


In [1]:
# ================================
# Run your experiments here
# ================================

# Suggested workflow
#
# 1. Use the same prompt.
# 2. Run Primary configuration.
# 3. Run Challenger A.
# 4. Run Challenger B.
# 5. If sampling is used:
#       Run AT LEAST 3 times.
#
# Paste or print representative outputs.

# The representative prompt from Step 3
representative_prompt = """Classify the following customer support ticket into one of these categories: 'Billing', 'Technical Support', 'Account Management', 'General Inquiry'.

Ticket: My internet isn't working after the update. I can't connect to any websites.
Category:"""

# ================================
# Run your experiments here
# ================================

print("--- Running Ticket Tagger Experiments ---")

# Primary Configuration (Greedy)
print("\n--- Primary Configuration (Greedy) ---")
print("Temperature: 0.0, Top-k: 1, Top-p: 0.0, Greedy")
primary_output = generate_with_params(representative_prompt, n_tokens=5, temperature=0.0, top_k=1, top_p=0.0)
print(f"Output: {primary_output}")

# Challenger A (Sampling) - Run 3 times
print("\n--- Challenger A (Sampling) ---")
print("Temperature: 0.1, Top-k: 5, Top-p: 0.1, Sampling")
for i in range(1, 4):
    challenger_a_output = generate_with_params(representative_prompt, n_tokens=5, temperature=0.1, top_k=5, top_p=0.1)
    print(f"Run {i} Output: {challenger_a_output}")

# Challenger B (Sampling) - Run 3 times
print("\n--- Challenger B (Sampling) ---")
print("Temperature: 0.5, Top-k: 40, Top-p: 0.9, Sampling")
for i in range(1, 4):
    challenger_b_output = generate_with_params(representative_prompt, n_tokens=5, temperature=0.5, top_k=40, top_p=0.9)
    print(f"Run {i} Output: {challenger_b_output}")

print("\nNote: The 'Penalty' parameter (e.g., repetition penalty) is typically applied to logits before softmax or sampling, which makes its implementation more complex in this manual probability-based sampling loop. For this exercise, we focused on Temperature, Top-k, and Top-p.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

--- Running Ticket Tagger Experiments ---

--- Primary Configuration (Greedy) ---
Temperature: 0.0, Top-k: 1, Top-p: 0.0, Greedy
Output: Classify the following customer support ticket into one of these categories: 'Billing', 'Technical Support', 'Account Management', 'General Inquiry'.

Ticket: My internet isn't working after the update. I can't connect to any websites.
Category: 'Customer Support'


--- Challenger A (Sampling) ---
Temperature: 0.1, Top-k: 5, Top-p: 0.1, Sampling
Run 1 Output: Classify the following customer support ticket into one of these categories: 'Billing', 'Technical Support', 'Account Management', 'General Inquiry'.

Ticket: My internet isn't working after the update. I can't connect to any websites.
Category: 'Customer Support'

Run 2 Output: Classify the following customer support ticket into one of these categories: 'Billing', 'Technical Support', 'Account Management', 'General Inquiry'.

Ticket: My internet isn't working after the update. I can't connect to

--- Environments ---
## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---:|
|Primary|1|the category is customer service but not in the list|0|the output is strikt and identical every time but not related to categories, i think it because the little value of top-k and top-p make it choose little next related tokens|
|Primary|2|the category is customer service but not in the list|0|the output is strikt and identical every time but not related to categories, i think it because the little value of top-k and top-p make it choose little next related tokens|
|Primary|3|the category is customer service but not in the list|0|the output is strikt and identical every time but not related to categories, i think it because the little value of top-k and top-p make it choose little next related tokens|
|Challenger A|1|consistently 'Technical Support' (in list)|1|even when minor sampling, it appeared strikt and consistently provides the same category and from the list because the suitable number of top-k and top-p and seems that the probability not closed|
|Challenger A|2|consistently 'Technical Support' (in list)|1|even when minor sampling, it appeared strikt and consistently provides the same category and from the list because the suitable number of top-k and top-p and seems that the probability not closed|
|Challenger A|3|consistently 'Technical Support' (in list)|1|even when minor sampling, it appeared strikt and consistently provides the same category and from the list because the suitable number of top-k and top-p and seems that the probability not closed|
|Challenger B|1|customer support(not in the list)|0|inconsistent outputs, often contain extra tokens because high top-k, category no in the list because high tempreature, top=k and top-p that result in more creativity and non- compliance with strict formatting|
|Challenger B|0|Billing(in the list)||when we use sampling it result in different results every run even if it was with the same parameters|
|Challenger B|3|customer support(not in the list)|0|inconsistent outputs, often contain extra tokens because high top-k, category no in the list because high tempreature, top=k and top-p that result in more creativity and non- compliance with strict formatting|

### Questions

- Which configuration won?
**Challenger A** won because it consistently provided a category from the predefined list, was deterministic, and met the strict format requirements of the success criteria.

- Why?
Challenger A, with its minor sampling (Temperature: 0.1, Top-k: 5, Top-p: 0.1), successfully found and consistently output 'Technical Support', which is a valid category. While Primary was deterministic, it failed to output a category from the allowed list. Challenger B failed due to inconsistency and non-compliance with the strict format.

- Did the evidence surprise you?
Yes, the evidence was surprising because I initially thought the Primary configuration (greedy decoding) would win due to its strictness. However, Challenger A demonstrated that a slight amount of sampling can be beneficial for selecting from a defined list while still maintaining determinism and strict format, something the overly rigid Primary configuration failed to do in this specific context.

# Reply Drafter

## Step 1 — Understand the Business Problem

### Success Criteria

**Professional, coherent, polite, natural.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
2. Is creativity helpful or harmful?
3. Should the model take risks?

Write your answers below.
1-identical outputs seems like robotic not natural and this is not out goal.

2-it will be helpful in many cases to avoid reppitition and ensure that replies are natural, but excessive creativity may lead to unrelated, unprofessional and off-topic replies

3-taking risks here will be harmful because it may result in inappropriate meannings and out goal is clear replies and polite communication so, the model should not take risks

--- Environments ---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---|---|---|
|Primary|0.7|40|0.9|0.0|Sampling|This configuration aims for a balance between naturalness and control. A moderate temperature and broad top-k/top-p allow for diverse and natural-sounding replies without sacrificing professionalism or coherence. It encourages varied phrasing suitable for customer service without being overly creative or off-topic.|
|Challenger A|0.1|5|0.1|0.0|Sampling|This more conservative configuration tests if very limited sampling can still produce natural-sounding replies. It aims for higher consistency but might result in more repetitive or robotic phrasing if the model doesn't have enough flexibility to choose varied tokens.|
|Challenger B|1.0|50|1.0|0.0|Sampling|This configuration maximizes randomness and creativity to demonstrate its potential harm for a professional communication task. A high temperature and wide top-k/top-p are expected to produce highly diverse, potentially informal, off-topic, or even inappropriate replies, violating the politeness and professionalism criteria.|

--- General ---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
Write a professional and polite customer service reply to the following customer complaint:

Customer Complaint: "I am very disappointed with the recent software update. It has caused several bugs, and my application crashes frequently. I need a solution urgently."

Reply:
```

In [2]:
# ================================
# Run your experiments here
# ================================

# Suggested workflow
#
# 1. Use the same prompt.
# 2. Run Primary configuration.
# 3. Run Challenger A.
# 4. Run Challenger B.
# 5. If sampling is used:
#       Run AT LEAST 3 times.
#
# Paste or print representative outputs.

# ================================
# Run your experiments here
# ================================
# The representative prompt from Step 3
representative_prompt = """Write a professional and polite customer service reply to the following customer complaint:
Customer Complaint: "I am very disappointed with the recent software update. It has caused several bugs, and my application crashes frequently. I need a solution urgently."
Reply:"""

# ================================
# Run your experiments here
# ================================

print("--- Running Reply Drafter Experiments ---")

print(f"Output: {primary_output}")
#primary configurations (sampling)
print("\n--- Primary Configuration (Sampling) ---")
print("Temperature: 0.7, Top-k: 40, Top-p: 0.9, Sampling")
for i in range(1, 4):
    primary_output = generate_with_params(representative_prompt, n_tokens=5, temperature=0.7, top_k=40, top_p=0.9)
    print(f"Run {i} Output: {primary_output}")
# Challenger A (Sampling) - Run 3 times
print("\n--- Challenger A (Sampling) ---")
print("Temperature: 0.1, Top-k: 5, Top-p: 0.1, Sampling")
for i in range(1, 4):
    challenger_a_output = generate_with_params(representative_prompt, n_tokens=5, temperature=0.1, top_k=5, top_p=0.1)
    print(f"Run {i} Output: {challenger_a_output}")

# Challenger B (Sampling) - Run 3 times
print("\n--- Challenger B (Sampling) ---")
print("Temperature: 0.5, Top-k: 40, Top-p: 0.9, Sampling")
for i in range(1, 4):
    challenger_b_output = generate_with_params(representative_prompt, n_tokens=5, temperature=0.5, top_k=40, top_p=0.9)
    print(f"Run {i} Output: {challenger_b_output}")

print("\nNote: The 'Penalty' parameter (e.g., repetition penalty) is typically applied to logits before softmax or sampling, which makes its implementation more complex in this manual probability-based sampling loop. For this exercise, we focused on Temperature, Top-k, and Top-p.")



--- Running Reply Drafter Experiments ---
Output: Classify the following customer support ticket into one of these categories: 'Billing', 'Technical Support', 'Account Management', 'General Inquiry'.

Ticket: My internet isn't working after the update. I can't connect to any websites.
Category: 'Customer Support'


--- Primary Configuration (Sampling) ---
Temperature: 0.7, Top-k: 40, Top-p: 0.9, Sampling
Run 1 Output: Write a professional and polite customer service reply to the following customer complaint:
Customer Complaint: "I am very disappointed with the recent software update. It has caused several bugs, and my application crashes frequently. I need a solution urgently."
Reply: "The latest software update
Run 2 Output: Write a professional and polite customer service reply to the following customer complaint:
Customer Complaint: "I am very disappointed with the recent software update. It has caused several bugs, and my application crashes frequently. I need a solution urgently."

--- Environments ---

## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---:|
|Primary|1|it results in the cause of the complaint|0|The model fails to switch context to generating a professional reply within the given `n_tokens=5`. The output is not a polite or professional response.|
|Primary|2|the model complete the complaint of the user||0|The model fails to switch context to generating a professional reply within the given `n_tokens=5`. The output is not a polite or professional response.|
|Primary|3|the answer is not related|0|The model fails to switch context to generating a professional reply within the given `n_tokens=5`. The output is not a polite or professional response.|
|Challenger A|1|Consistently repeats a part of the customer's complaint ('I am very disappointed')|0|Despite high consistency, the model fails to generate a professional reply, instead repeating part of the complaint. The limited tokens (`n_tokens=5`) prevent it from initiating an actual reply.|
|Challenger A|2|Consistently repeats a part of the customer's complaint ('I am very disappointed')|0|Despite high consistency, the model fails to generate a professional reply, instead repeating part of the complaint. The limited tokens (`n_tokens=5`) prevent it from initiating an actual reply.|
|Challenger A|3|Consistently repeats a part of the customer's complaint ('I am very disappointed')|0|Despite high consistency, the model fails to generate a professional reply, instead repeating part of the complaint. The limited tokens (`n_tokens=5`) prevent it from initiating an actual reply.|
|Challenger B|1|Starts with irrelevant text ('I have had a') not a professional reply|0|High randomness leads to varied but inappropriate outputs for a professional customer service reply. The model doesn't switch to the 'reply' context effectively within the given `n_tokens=5`.|
|Challenger B|2|Starts with irrelevant text ('I have not received') not a professional reply|0|High randomness leads to varied but inappropriate outputs for a professional customer service reply. The model doesn't switch to the 'reply' context effectively within the given `n_tokens=5`.|
|Challenger B|3|Starts with irrelevant text ('I received the software') not a professional reply|0|High randomness leads to varied but inappropriate outputs for a professional customer service reply. The model doesn't switch to the 'reply' context effectively within the given `n_tokens=5`.|


|---|---:|---|---:|---|
|Primary|1||0|the high teampreture make it more randomized and inapropriate in the replies and and high top-k and top-p that results in unrelated answers|
|Primary|2|0|the high teampreture make it more randomized and inapropriate in the replies and and high top-k and top-p that results in unrelated answers|
|Primary|3|the answer no related|0|the high teampreture make it more randomized and inapropriate in the replies and and high top-k and top-p that results in unrelated answers|
|Challenger A|1||||
|Challenger A|2||||
|Challenger A|3||||
|Challenger B|1||||
|Challenger B|2||||
|Challenger B|3||||

### Questions

- Which configuration won?
None of the configurations effectively 'won' for the Reply Drafter task with the current `n_tokens=5` limitation. All configurations scored 0 as they failed to generate a professional, coherent, polite, and natural *reply* to the customer complaint.

- Why?
The primary reason for failure across all configurations is the extremely low `n_tokens` (set to 5). This severely limits the model's ability to transition from processing the prompt to generating a meaningful response that satisfies the 'Reply:' instruction and the task's success criteria. Instead, the model either continues the customer's complaint or generates unrelated internal monologue.

- Did the evidence surprise you?
The consistent failure to generate even the beginning of a proper reply, especially for Challenger A which was expected to be more controlled, was somewhat surprising. It highlights how crucial `n_tokens` can be for tasks requiring a context shift and a minimal amount of output to be considered successful. The short output length masked any nuanced differences in how Temperature, Top-k, and Top-p might otherwise influence the *style* of a generated reply.

# Campaign Brainstormer

## Step 1 — Understand the Business Problem

### Success Criteria

**Creative, diverse, non-repetitive.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical? no because my goal is to be non repetitive
2. Is creativity helpful or harmful?helpful because the success criteria is
3. Should the model take risks?yes because taking risks make it more diverse

Write your answers below.


--- Environments ---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---|---|---|
|Primary|0.9|5|0.8|0|sampling|This configuration aims for a highly creative, non-repetitive, and diverse output. A high temperature and moderate top-k/top-p allow the model to explore a wide range of tokens, fostering originality while maintaining some coherence.|
|Challenger A|0.5|3|0.1|0|sampling|This configuration tests a more controlled approach to creativity. A lower temperature and very restricted top-k/top-p limit the model's choices, potentially leading to less diverse but more focused ideas, which might still be valuable if consistency is also desired.|
|Challenger B|1.0|50|1.0|0|Sampling|This configuration maximizes randomness and creativity. A very high temperature and unrestricted top-k/top-p allow the model to explore the broadest possible range of token sequences, aiming for highly diverse and unconventional ideas to fulfill the 'creative' and 'diverse' success criteria. This might come at the cost of some coherence but pushes the boundaries for brainstorming.|

--- General ---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
Brainstorm 10 creative, diverse, and non-repetitive marketing campaign ideas for a new sustainable fashion brand targeting Gen Z. Focus on digital channels.

Ideas:
```

In [4]:
# ================================
# Run your experiments here
# ================================

# Suggested workflow
#
# 1. Use the same prompt.
# 2. Run Primary configuration.
# 3. Run Challenger A.
# 4. Run Challenger B.
# 5. If sampling is used:
#       Run AT LEAST 3 times.
#
# Paste or print representative outputs.


# ================================
# Run your experiments here
# ================================
# The representative prompt from Step 3
representative_prompt = """Brainstorm 10 creative, diverse, and non-repetitive marketing campaign ideas for a new sustainable fashion brand targeting Gen Z. Focus on digital channels.
Ideas:"""

# ================================
# Run your experiments here
# ================================

print("--- Running Ticket Tagger Experiments ---")

#primary (sampling)- Run 3 times
print("\n--- Primary Configuration (Sampling) ---")
print("Temperature: 0.9, Top-k: 5, Top-p: 0.8, Sampling")
for i in range(1, 4):
    primary_output = generate_with_params(representative_prompt, n_tokens=5, temperature=0.9, top_k=5, top_p=0.8)
    print(f"Run {i} Output: {primary_output}")
# Challenger A (Sampling) - Run 3 times
print("\n--- Challenger A (Sampling) ---")
print("Temperature: 0.1, Top-k: 5, Top-p: 0.1, Sampling")
for i in range(1, 4):
    challenger_a_output = generate_with_params(representative_prompt, n_tokens=5, temperature=0.1, top_k=5, top_p=0.1)
    print(f"Run {i} Output: {challenger_a_output}")

# Challenger B (Sampling) - Run 3 times
print("\n--- Challenger B (Sampling) ---")
print("Temperature: 0.5, Top-k: 40, Top-p: 0.9, Sampling")
for i in range(1, 4):
    challenger_b_output = generate_with_params(representative_prompt, n_tokens=5, temperature=0.5, top_k=40, top_p=0.9)
    print(f"Run {i} Output: {challenger_b_output}")

print("\nNote: The 'Penalty' parameter (e.g., repetition penalty) is typically applied to logits before softmax or sampling, which makes its implementation more complex in this manual probability-based sampling loop. For this exercise, we focused on Temperature, Top-k, and Top-p.")


--- Running Ticket Tagger Experiments ---

--- Primary Configuration (Sampling) ---
Temperature: 0.9, Top-k: 5, Top-p: 0.8, Sampling
Run 1 Output: Brainstorm 10 creative, diverse, and non-repetitive marketing campaign ideas for a new sustainable fashion brand targeting Gen Z. Focus on digital channels.
Ideas:
- Create a brand
Run 2 Output: Brainstorm 10 creative, diverse, and non-repetitive marketing campaign ideas for a new sustainable fashion brand targeting Gen Z. Focus on digital channels.
Ideas:
1. Create a
Run 3 Output: Brainstorm 10 creative, diverse, and non-repetitive marketing campaign ideas for a new sustainable fashion brand targeting Gen Z. Focus on digital channels.
Ideas:
-Create a new

--- Challenger A (Sampling) ---
Temperature: 0.1, Top-k: 5, Top-p: 0.1, Sampling
Run 1 Output: Brainstorm 10 creative, diverse, and non-repetitive marketing campaign ideas for a new sustainable fashion brand targeting Gen Z. Focus on digital channels.
Ideas:
- Create a brand
Run 2 Output:

--- Environments ---

## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---:|
|Primary|1|Starts with 'Create a brand'|0|The output is very short because`n_tokens=5`, making it impossible to increase creativity, diversity, or non-repetitiveness. It's merely the beginning of a generic phrase.|
|Primary|2|Starts with 'Create a'|0|The output is very short because `n_tokens=5`, making it impossible to increase creativity, diversity, or non-repetitiveness. It's merely the beginning of a generic phrase.|
|Primary|3|Starts with 'Create a new'|0|The output is very short because `n_tokens=5`, making it impossible to increase creativity, diversity, or non-repetitiveness. It's merely the beginning of a generic phrase.|
|Challenger A|1|Starts with 'Create a brand'|0|The output is identical across runs because of low tempreature and very short due to `n_tokens=5`, making it impossible to assess creativity, diversity, or non-repetitiveness. It's merely the beginning of a generic phrase.|
|Challenger A|2|Starts with 'Create a brand'|0|The output is identical across runs because of low tempreature and extremely short due to `n_tokens=5`, making it impossible to assess creativity, diversity, or non-repetitiveness. It's merely the beginning of a generic phrase.|
|Challenger A|3|Starts with 'Create a brand'|0|The output is identical across runs because of low tempreature and extremely short due to `n_tokens=5`, making it impossible to assess creativity, diversity, or non-repetitiveness. It's merely the beginning of a generic phrase.|
|Challenger B|1|Starts with 'A new fashion brand'|0|The output is extremely short due to `n_tokens=5`, making it impossible to assess creativity, diversity, or non-repetitiveness. It's merely the beginning of a generic phrase.|
|Challenger B|2|Starts with 'Create a'|0|The output is very short due to `n_tokens=5`, making it impossible to assess creativity, diversity, or non-repetitiveness. It's merely the beginning of a generic phrase.|
|Challenger B|3|Starts with 'Create a new'|0|The output is very short due to `n_tokens=5`, making it impossible to assess creativity, diversity, or non-repetitiveness. It's merely the beginning of a generic phrase.|

### Questions

- Which configuration won?
None of the configurations can be considered a 'winner' with the current `n_tokens=5` limitation. All outputs are too short to demonstrate creativity, diversity, or non-repetitiveness, consistently scoring 0 as they fail to generate any meaningful campaign ideas.

- Why?
The primary reason for failure across all configurations is the extremely low `n_tokens` (set to 5). This severely limits the model's ability to generate more than the first few tokens of an idea, preventing any assessment of the desired success criteria like creativity, diversity, or non-repetitiveness. The model doesn't have enough space to even start brainstorming.

- Did the evidence surprise you?
No, given the prior experience with the 'Reply Drafter' product, it's consistent that such a low `n_tokens` would prevent any configuration from succeeding in tasks requiring more elaborate or creative output. This further emphasizes the critical role of output length in LLM performance for different tasks.

# 📝 Part C — The Decoding Playbook

Imagine a new engineer joins your team tomorrow.

They should be able to use this page **without reading the rest of the notebook.**

## Final Recommendations

|Feature|Recommended Configuration|Reason|
|---|---|---|
|Ticket Tagger|tempreature: 0.1, top-k:\t5\ttop-p:0.1, penality:0.0, sampling|because this configuration meet the succes criteria, deterministic and strikt formatting|
|Reply Drafter|temp:0.7, top-k:40, top-p:\t0.9\tpenality:0.0|middle tepreature so answers will be not identical and have a little creativity and will related because of high top-p|
|Campaign Brainstormer|Re-run experiments with `n_tokens` > 5|All configurations failed to generate meaningful output due to `n_tokens=5`. Higher `n_tokens` is crucial for assessing creativity, diversity, and non-repetitiveness, as brainstorming requires more extensive output.|

---

## House Rule #1

Example:

> Use deterministic decoding when output format is strict.

Write your own:

---

## House Rule #2

Example:

> Use sampling only when diversity creates business value.

Write your own.

---

## Biggest Limitation

Choose one and explain why it matters.

- GPT-2 is a small model
- Small sample size
- Subjective scoring
- Other

# ✅ Submission Checklist

- [ ] I defined success criteria before testing.
- [ ] I designed three configurations for every feature.
- [ ] Every stochastic configuration was executed at least three times.
- [ ] I scored outputs before deciding the winner.
- [ ] My final playbook is self-contained.
- [ ] All notebook cells are executed.

---

## ⭐ Bonus (Optional)

Complete ONE:

- Compare Greedy vs Sampling visually.
- Plot the effect of different temperatures.
- Demonstrate reproducibility using random seeds.
- Show a challenger configuration outperforming your primary recommendation.
